# Optimization Modelling

We will begin our modelling by importing our pre-processed data.

In [ ]:
import pandas as pd
import numpy as np

avg_income = pd.read_csv('../data/task_1/avg_income_nyc.csv')
child_care = pd.read_csv('../data/task_1/child_care_regulated_nyc.csv')
employment_rate = pd.read_csv('../data/task_1/employment_rate_nyc.csv')
population = pd.read_csv('../data/task_1/population_nyc.csv')

## Modelling Assumptions

For this model, we assume that each zip code only facilitates the population within its own boundary and does not serve children from neighbouring zip codes. Specifically, in order for a child to enrol in a child care facility, their home address must match the zip code of that facility.

We also assume that all facilities are valid to expand (even if status is Suspended) because if the model indicates that we need to build more this facility its worth keep it running.

## Child Care Desert Identification

We identify zip codes that are currently child care deserts based on demand classification and capacity thresholds.

### Step 1: Extract Child Population by Zip Code

We extract two age groups from the population dataset:

- **Ages 0–5** (`pop_0_5`): relevant for the NYC Under-5 Policy threshold
- **Ages 6–12** (`pop_6_12`): combined with ages 0–5 to form the full child care age range

$$\text{pop\_2wk\_12yr} = \text{pop\_0\_5} + \text{pop\_6\_12}$$

In [ ]:
# Rename columns for clarity: '-5' = ages 0-5, '6-12' = ages 6-12
pop = population.rename(columns={'-5': 'pop_0_5', '6-12': 'pop_6_12'})[['zipcode', 'pop_0_5', 'pop_6_12']]

# pop_2wk_12yr = pop_0_5 + pop_6_12
# This represents all children who may need child care (ages ~2 weeks to 12 years)
pop['pop_2wk_12yr'] = pop['pop_0_5'] + pop['pop_6_12']

pop.head()

,zipcode,pop_0_5,pop_6_12,pop_2wk_12yr
0,10001,744,1255,1999
1,10002,2142,4645,6787
2,10003,1440,1510,2950
3,10004,433,262,695
4,10005,484,318,802


### Step 2: Aggregate Child Care Slots by Zip Code

Each facility record contains capacity broken down by age group. We derive two slot totals per zip:

- **Total slots**: sum of `total_capacity` across all facilities in the zip
- **Under-5 slots**: sum of infant + toddler + preschool capacity, covering ages 0–5

$$\text{slots\_0\_5} = \text{infant\_capacity} + \text{toddler\_capacity} + \text{preschool\_capacity}$$

$$\text{total\_slots}_z = \sum_{\text{facility} \in z} \text{total\_capacity}, \quad \text{slots\_0\_5}_z = \sum_{\text{facility} \in z} \text{slots\_0\_5}$$

In [ ]:
# Under-5 slots = infant + toddler + preschool capacity per facility
child_care['slots_0_5'] = (
    child_care['infant_capacity'] +
    child_care['toddler_capacity'] +
    child_care['preschool_capacity']
)

# Aggregate to zip code level: sum total_capacity and slots_0_5 across all facilities
slots = child_care.groupby('zipcode').agg(
    total_slots=('total_capacity', 'sum'),
    slots_0_5=('slots_0_5', 'sum')
).reset_index()

slots.head()

,zipcode,total_slots,slots_0_5
0,10001,609,0
1,10002,4729,18
2,10003,1995,0
3,10004,263,0
4,10005,39,0


### Step 3: Merge All Datasets

We join population, slots, income, and employment rate on `zipcode`. Zip codes with no registered facilities receive 0 slots.

In [ ]:
df = pop.merge(slots, on='zipcode', how='left')
df = df.merge(avg_income.rename(columns={'average income': 'avg_income'}), on='zipcode', how='left')
df = df.merge(employment_rate.rename(columns={'employment rate': 'emp_rate'}), on='zipcode', how='left')

# Zip codes with no child care facilities will have NaN slots — treat as 0
df[['total_slots', 'slots_0_5']] = df[['total_slots', 'slots_0_5']].fillna(0)

print(f"Total zip codes in dataset: {len(df)}")
df.head()

Total zip codes in dataset: 180


,zipcode,pop_0_5,pop_6_12,pop_2wk_12yr,total_slots,slots_0_5,avg_income,emp_rate
0,10001,744,1255,1999,609.0,0.0,102878.033603,0.595097
1,10002,2142,4645,6787,4729.0,18.0,59604.041165,0.520662
2,10003,1440,1510,2950,1995.0,0.0,114273.049645,0.497244
3,10004,433,262,695,263.0,0.0,132004.310345,0.506661
4,10005,484,318,802,39.0,0.0,121437.713311,0.665833


### Step 4: Demand Classification

Each zip code is classified as **High-Demand** or **Normal-Demand** based on income and employment:

$$\text{High-Demand} \iff \text{Avg Income} \leq \$60{,}000 \;\lor\; \text{Employment Rate} \geq 60\%$$

$$\text{Normal-Demand} \iff \text{Avg Income} > \$60{,}000 \;\land\; \text{Employment Rate} < 60\%$$

High-demand areas face greater pressure on child care availability — lower income limits family ability to pay for private care, while high employment means more parents are in the workforce and actively need external child care.

In [ ]:
# High-Demand: avg_income <= $60,000 OR emp_rate >= 0.60
# Normal-Demand: avg_income > $60,000 AND emp_rate < 0.60
df['demand'] = df.apply(
    lambda r: 'High' if (r['avg_income'] <= 60_000 or r['emp_rate'] >= 0.60) else 'Normal',
    axis=1
)

print(df['demand'].value_counts().to_string())
df[['zipcode', 'avg_income', 'emp_rate', 'demand']].head(10)

demand
High      97
Normal    83


,zipcode,avg_income,emp_rate,demand
0,10001,102878.033603,0.595097,Normal
1,10002,59604.041165,0.520662,High
2,10003,114273.049645,0.497244,Normal
3,10004,132004.310345,0.506661,Normal
4,10005,121437.713311,0.665833,High
5,10006,126377.118644,0.631692,High
6,10007,138853.904282,0.528910,Normal
7,10009,77133.233533,0.514567,Normal
8,10010,116272.698810,0.492749,Normal
9,10011,120420.792079,0.557000,Normal


### Step 5: Desert Identification

A zip code is flagged as a **child care desert** if it fails **any** of the following three conditions:

**Condition 1 — Total Capacity (demand-adjusted):**

$$\text{Desert if } \begin{cases} \text{total\_slots} < 0.5 \times \text{pop\_2wk\_12yr} & \text{if High-Demand} \\ \text{total\_slots} < 1/3 \times \text{pop\_2wk\_12yr} & \text{if Normal-Demand} \end{cases}$$

**Condition 2 — NYC Under-5 Policy (applies to all zips):**

$$\text{Desert if } \text{slots\_0\_5} < 2/3 \times \text{pop\_0\_5}$$

A zip is a desert if Condition 1 **or** Condition 2 is met:

$$\text{is\_desert} = \text{fail\_total} \lor \text{fail\_under5}$$

In [ ]:
THRESHOLD_HIGH   = 1/2  # High-demand: total_slots < 0.5   * pop_2wk_12yr  → desert
THRESHOLD_NORMAL = 1/3  # Normal-demand: total_slots < 0.333 * pop_2wk_12yr  → desert
THRESHOLD_UNDER5 = 2/3  # All zips: slots_0_5 < 0.667 * pop_0_5             → desert

# Required total slot threshold per zip (depends on demand class)
df['total_threshold'] = df.apply(
    lambda r: THRESHOLD_HIGH * r['pop_2wk_12yr'] if r['demand'] == 'High'
              else THRESHOLD_NORMAL * r['pop_2wk_12yr'],
    axis=1
)

# Condition 1: total_slots <= threshold  →  fail_total = True
df['fail_total']  = df['total_slots'] <= df['total_threshold']

# Condition 2: slots_0_5 <= 0.667 * pop_0_5  →  fail_under5 = True
df['fail_under5'] = df['slots_0_5'] <= THRESHOLD_UNDER5 * df['pop_0_5']

# Desert = fails either condition
df['is_desert'] = (df['fail_total'] | df['fail_under5']) & (df['pop_2wk_12yr'] > 0)

print(f"Zips failing total capacity check : {df['fail_total'].sum()}")
print(f"Zips failing under-5 policy check : {df['fail_under5'].sum()}")
print(f"Total desert zip codes            : {df['is_desert'].sum()}")

Zips failing total capacity check : 162
Zips failing under-5 policy check : 180
Total desert zip codes            : 179


**Note on desert counts:** 180 zips fail the under-5 check but only 179 are classified as deserts. The discrepancy comes from zip **11005**, which has zero children (`pop_2wk_12yr = 0`). Because `slots_0_5 = 0` and `pop_0_5 = 0`, the under-5 condition trivially fires (`0 ≤ 0`), but the zip is excluded from deserts by the `& (pop_2wk_12yr > 0)` guard — there is no child population to serve.

### Step 6: Compute Slot Deficits

For each desert zip code we compute how many additional slots are needed to meet each threshold:

$$\text{Total Slot Deficit} = \max\!\left(0,\; \text{total\_threshold} - \text{total\_slots}\right)$$

$$\text{Under-5 Slot Deficit} = \max\!\left(0,\; 0.667 \times \text{pop\_0\_5} - \text{slots\_0\_5}\right)$$

A deficit of 0 on one metric means that particular condition was already satisfied — the zip is still a desert due to failing the other condition.

In [ ]:
# Total Slot Deficit = max(0, total_threshold - total_slots)
df['total_slot_deficit'] = np.ceil((df['total_threshold'] - df['total_slots']).clip(lower=0)).astype(int)

# Under-5 Slot Deficit = max(0, 0.667 * pop_0_5 - slots_0_5)
df['under5_slot_deficit'] = np.ceil((THRESHOLD_UNDER5 * df['pop_0_5'] - df['slots_0_5']).clip(lower=0)).astype(int)

df[['zipcode', 'total_slots', 'total_threshold', 'total_slot_deficit',
    'slots_0_5', 'under5_slot_deficit']].head(10)

,zipcode,total_slots,total_threshold,total_slot_deficit,slots_0_5,under5_slot_deficit
0,10001,609.0,666.333333,58,0.0,496
1,10002,4729.0,3393.500000,0,18.0,1410
2,10003,1995.0,983.333333,0,0.0,960
3,10004,263.0,231.666667,0,0.0,289
4,10005,39.0,401.000000,362,0.0,323
5,10006,156.0,130.500000,0,14.0,72
6,10007,284.0,400.333333,117,0.0,404
7,10009,1784.0,1490.333333,0,18.0,1246
8,10010,234.0,1162.666667,929,0.0,948
9,10011,1956.0,1030.333333,0,42.0,764


### Step 7: Output — Desert Zip Codes

In [ ]:
# Filter to desert zips only and select output columns
deserts = df[df['is_desert']][['zipcode', 'demand', 'total_slot_deficit', 'under5_slot_deficit']].copy()
deserts.columns = ['Zip Code', 'Demand Classification', 'Total Slot Deficit', 'Under-5 Slot Deficit']
deserts = deserts.sort_values('Zip Code').reset_index(drop=True)

print(f"Total desert zip codes identified: {len(deserts)}")
deserts

Total desert zip codes identified: 179


,Zip Code,Demand Classification,Total Slot Deficit,Under-5 Slot Deficit
0,10001,Normal,58,496
1,10002,High,0,1410
2,10003,Normal,0,960
3,10004,Normal,0,289
4,10005,High,362,323
...,...,...,...,...
174,11691,High,4431,4032
175,11692,High,1553,1139
176,11693,Normal,265,454
177,11694,Normal,405,911


### Non-Desert Zip Codes

Zip codes that meet **all** capacity conditions and are therefore adequately served.

In [ ]:
non_deserts = df[~df['is_desert']][['zipcode', 'demand', 'total_slots', 'total_threshold', 'slots_0_5']].copy()
non_deserts.columns = ['Zip Code', 'Demand Classification', 'Total Slots', 'Required Total Slots', 'Under-5 Slots']
non_deserts = non_deserts.sort_values('Zip Code').reset_index(drop=True)

print(f"Total non-desert zip codes: {len(non_deserts)}")
non_deserts
#It has no kids lol

Total non-desert zip codes: 1


,Zip Code,Demand Classification,Total Slots,Required Total Slots,Under-5 Slots
0,11005,High,0.0,0.0,0.0


---

## Optimization Model

We now formulate a Gurobi optimization model to determine how to best eliminate child care deserts across NYC zip codes. The model decides whether to **expand existing facilities** and/or **build new facilities** of different sizes in each desert zip code.

### Step 1: Decision Variables

We define four types of decision variables, one per desert zip code $z \in \mathcal{Z}$ and per existing facility $f \in \mathcal{F}_z$.

---

**Variable 1a — Facility Expansion (Ages 0–5)** $x_f^{05}$

For each existing facility $f$ in a desert zip, $x_f^{05}$ is the number of additional slots added for children aged 0–5:

$$x_f^{05} \in \mathbb{Z}_{\geq 0}, \quad \forall\, f \in \mathcal{F}_z,\; z \in \mathcal{Z}$$

---

**Variable 1b — Facility Expansion (Ages 5–12)** $x_f^{512}$

For each existing facility $f$ in a desert zip, $x_f^{512}$ is the number of additional slots added for children aged 5–12:

$$x_f^{512} \in \mathbb{Z}_{\geq 0}, \quad \forall\, f \in \mathcal{F}_z,\; z \in \mathcal{Z}$$

---

**Variable 2 — New Facility Construction** $y_z^S,\; y_z^M,\; y_z^L$

For each desert zip $z$, these count how many new Small, Medium, or Large facilities to build:

$$y_z^S,\; y_z^M,\; y_z^L \;\in \mathbb{Z}_{\geq 0}, \quad \forall\, z \in \mathcal{Z}$$

---

**Variable 3a — New Build Slots (Ages 0–5)** $n_z^{05}$

For each desert zip $z$, $n_z^{05}$ is the number of slots in new facilities dedicated to children aged 0–5:

$$n_z^{05} \in \mathbb{Z}_{\geq 0}, \quad \forall\, z \in \mathcal{Z}$$

---

**Variable 3b — New Build Slots (Ages 5–12)** $n_z^{512}$

For each desert zip $z$, $n_z^{512}$ is the number of slots in new facilities dedicated to children aged 5–12:

$$n_z^{512} \in \mathbb{Z}_{\geq 0}, \quad \forall\, z \in \mathcal{Z}$$


In [ ]:
import gurobipy as gp
from gurobipy import GRB

# ── Index sets ────────────────────────────────────────────────────────────────
desert_zips = df[df['is_desert']]['zipcode'].tolist()

desert_facilities = child_care[child_care['zipcode'].isin(desert_zips)].copy()

# Drop facilities with total_capacity == 0 — all sub-capacities are 0, likely a data error
zero_cap_count = (desert_facilities['total_capacity'] == 0).sum()
desert_facilities = desert_facilities[desert_facilities['total_capacity'] > 0].copy()
print(f"Facilities in desert zips          : {len(desert_facilities) + zero_cap_count}")
print(f"  → dropped (total_capacity == 0)  : {zero_cap_count}")
print(f"  → remaining                      : {len(desert_facilities)}")

# Current capacity lookup: {facility_id: n_f}
cap_lookup = desert_facilities.set_index('facility_id')['total_capacity'].to_dict()

# ── Gurobi model ──────────────────────────────────────────────────────────────
m = gp.Model("ChildCareDesert")
m.setParam('OutputFlag', 0)

# ── Decision Variable 1a: x_05[f] — 0-5 slots added to existing facility f ──
# x_f^05 ∈ Z≥0  for each facility f in a desert zip
x_05 = m.addVars(list(cap_lookup.keys()), vtype=GRB.INTEGER, lb=0, name="x_05")

# ── Decision Variable 1b: x_512[f] — 5-12 slots added to existing facility f ─
# x_f^512 ∈ Z≥0  for each facility f in a desert zip
x_512 = m.addVars(list(cap_lookup.keys()), vtype=GRB.INTEGER, lb=0, name="x_512")

# ── Decision Variable 2: y[z, size] — new facilities built in zip z ──────────
# y_z^S, y_z^M, y_z^L ∈ Z≥0  for each desert zip z
y = {}
for size in ['S', 'M', 'L']:
    y[size] = m.addVars(desert_zips, vtype=GRB.INTEGER, lb=0, name=f"y_{size}")

# ── Decision Variable 3a: n_05[z] — 0-5 slots from new builds in zip z ───────
# n_z^05 ∈ Z≥0  for each desert zip z
n_05 = m.addVars(desert_zips, vtype=GRB.INTEGER, lb=0, name="n_05")

# ── Decision Variable 3b: n_512[z] — 5-12 slots from new builds in zip z ─────
# n_z^512 ∈ Z≥0  for each desert zip z
n_512 = m.addVars(desert_zips, vtype=GRB.INTEGER, lb=0, name="n_512")

m.update()
print(f"\nDecision variables defined:")
print(f"  x_05 : {len(x_05):>5} variables  (0-5  expansion slots per existing facility)")
print(f"  x_512: {len(x_512):>5} variables  (5-12 expansion slots per existing facility)")
print(f"  y_S  : {len(y['S']):>5} variables  (new small  facilities per desert zip)")
print(f"  y_M  : {len(y['M']):>5} variables  (new medium facilities per desert zip)")
print(f"  y_L  : {len(y['L']):>5} variables  (new large  facilities per desert zip)")
print(f"  n_05 : {len(n_05):>5} variables  (0-5  slots from new builds per desert zip)")
print(f"  n_512: {len(n_512):>5} variables  (5-12 slots from new builds per desert zip)")
print(f"\nTotal variables: {m.NumVars}")


Facilities in desert zips          : 7740
  → dropped (total_capacity == 0)  : 1
  → remaining                      : 7739
Set parameter Username
Set parameter LicenseID to value 2773933
Academic license - for non-commercial use only - expires 2027-02-02

Decision variables defined:
  x_05 :  7739 variables  (0-5  expansion slots per existing facility)
  x_512:  7739 variables  (5-12 expansion slots per existing facility)
  y_S  :   179 variables  (new small  facilities per desert zip)
  y_M  :   179 variables  (new medium facilities per desert zip)
  y_L  :   179 variables  (new large  facilities per desert zip)
  n_05 :   179 variables  (0-5  slots from new builds per desert zip)
  n_512:   179 variables  (5-12 slots from new builds per desert zip)

Total variables: 16373


### Step 2: Objective Function — Minimize Total Cost

We minimize the sum of three cost components:

$$\min \quad C_{\text{exp}} + C_{\text{build}} + C_{\text{equip}}$$

---

**A. Expansion Cost** $C_{\text{exp}}$

The cost of expanding facility $f$ is a **step function** of the total slots added $x_f = x_f^{05} + x_f^{512}$, with two rates depending on whether the expansion falls in the cheap or expensive tier.

Define per facility:

$$\delta_1^f = \text{slots in the cheap tier}, \quad \delta_2^f = \text{slots in the expensive tier}$$

such that $x_f = \delta_1^f + \delta_2^f$.

A binary $w_f \in \{0,1\}$ selects the active tier ($w_f = 0$: cheap when $x_f < n_f$, $w_f = 1$: expensive when $x_f \geq n_f$). Let $M_f = \min(1.2\,n_f,\;500)$ be the expansion upper bound (from Constraint 5).

$$0 \leq \delta_1^f \leq n_f \cdot (1 - w_f)$$

$$n_f \cdot w_f \leq \delta_2^f \leq M_f \cdot w_f$$

The lower bound on $\delta_2^f$ ensures that once the expensive tier is active ($w_f = 1$), the expansion is at least $n_f$ — correctly enforcing that the tier switch only happens when $x_f \geq n_f$.

The per-facility cost is then:

$$C_{\text{exp}}(f) = c_1 \cdot \delta_1^f + c_2 \cdot \delta_2^f, \qquad
c_1 = \frac{10{,}000 + 200\,n_f}{n_f}, \quad c_2 = \frac{20{,}000 + 200\,n_f}{n_f}$$

$$C_{\text{exp}} = \sum_{z \in \mathcal{Z}} \sum_{f \in \mathcal{F}_z} \left(c_1 \cdot \delta_1^f + c_2 \cdot \delta_2^f\right)$$

---

**B. New Construction Cost** $C_{\text{build}}$

Fixed cost per new facility, multiplied by how many of each size are built in each zip:

| Size   | Slots | Cost per facility |
|--------|-------|-------------------|
| Small  | 100   | \$65,000          |
| Medium | 200   | \$95,000          |
| Large  | 400   | \$115,000         |

$$C_{\text{build}} = \sum_{z \in \mathcal{Z}} \left(65{,}000\, y_z^S + 95{,}000\, y_z^M + 115{,}000\, y_z^L\right)$$

---

**C. Equipment Surcharge** $C_{\text{equip}}$

A flat \$100 surcharge applies to every slot newly allocated to children aged 0–5 — both from facility expansions and new builds:

$$C_{\text{equip}} = 100 \sum_{z \in \mathcal{Z}} \left(\sum_{f \in \mathcal{F}_z} x_f^{05} + n_z^{05}\right)$$

In [ ]:
# ── Cost parameters ───────────────────────────────────────────────────────────
COST_S = 65_000    # $ per new small  facility (100 slots)
COST_M = 95_000    # $ per new medium facility (200 slots)
COST_L = 115_000   # $ per new large  facility (400 slots)

CAP_S  = 100       # total slots in a new small  facility
CAP_M  = 200       # total slots in a new medium facility
CAP_L  = 400       # total slots in a new large  facility

EQUIP_COST = 100   # $ per 0-5 slot added (expansion or new build)

# ── A. Expansion Cost: two-tier step function with binary tier selection ──────
# w[f] ∈ {0,1}: 0 → cheap tier (x_f < n_f), 1 → expensive tier (x_f ≥ n_f)
# M_f = min(1.2·n_f, 500): expansion upper bound (matches Constraint 5)
#
# δ₁[f]: slots in the cheap tier,     0       ≤ δ₁ ≤ n_f · (1 − w_f)
# δ₂[f]: slots in the expensive tier, n_f·w_f ≤ δ₂ ≤ M_f · w_f
# x_f = δ₁[f] + δ₂[f]   (enforced as equality constraint)
#
# Lower bound on δ₂ ensures that when w=1, x_f ≥ n_f (tier only flips at boundary)
#
# C_exp(f) = c1·δ₁[f] + c2·δ₂[f]

cap_lookup = desert_facilities.set_index('facility_id')['total_capacity'].to_dict()
fac_by_zip = desert_facilities.groupby('zipcode')['facility_id'].apply(list).to_dict()

w  = m.addVars(list(cap_lookup.keys()), vtype=GRB.BINARY,      name="w")
d1 = m.addVars(list(cap_lookup.keys()), vtype=GRB.CONTINUOUS, lb=0, name="d1")
d2 = m.addVars(list(cap_lookup.keys()), vtype=GRB.CONTINUOUS, lb=0, name="d2")

for f, n_f in cap_lookup.items():
    M_f = min(1.2 * n_f, 500)   # per-facility expansion upper bound

    # δ₁ ≤ n_f · (1 − w_f)  →  only active (and ≤ n_f) when on cheap tier
    m.addConstr(d1[f] <= n_f * (1 - w[f]),  name=f"d1_ub_{f}")
    # n_f · w_f ≤ δ₂ ≤ M_f · w_f  →  when expensive tier active, x_f ≥ n_f
    m.addConstr(d2[f] >= n_f * w[f],         name=f"d2_lb_{f}")
    m.addConstr(d2[f] <= M_f * w[f],         name=f"d2_ub_{f}")
    # x_f = δ₁ + δ₂
    m.addConstr(x_05[f] + x_512[f] == d1[f] + d2[f], name=f"x_split_{f}")

C_exp = gp.quicksum(
    (10_000 + 200 * cap_lookup[f]) / cap_lookup[f] * d1[f]
    + (20_000 + 200 * cap_lookup[f]) / cap_lookup[f] * d2[f]
    for f in cap_lookup
)

# ── B. New Construction Cost: C_build ─────────────────────────────────────────
C_build = gp.quicksum(
    COST_S * y['S'][z] + COST_M * y['M'][z] + COST_L * y['L'][z]
    for z in desert_zips
)

# ── C. Equipment Surcharge: C_equip ───────────────────────────────────────────
C_equip = EQUIP_COST * gp.quicksum(
    gp.quicksum(x_05[f] for f in fac_by_zip.get(z, []))
    + n_05[z]
    for z in desert_zips
)

# ── Set Objective ─────────────────────────────────────────────────────────────
m.setObjective(C_exp + C_build + C_equip, GRB.MINIMIZE)

m.update()
print("Objective function set: minimize C_exp + C_build + C_equip")
print(f"  Tier binary vars (w): {len(w)},  δ₁ vars: {len(d1)},  δ₂ vars: {len(d2)}")

Objective function set: minimize C_exp + C_build + C_equip
  Tier binary vars (w): 7739,  δ₁ vars: 7739,  δ₂ vars: 7739


### Step 3: Constraints

For each desert zip $z \in \mathcal{Z}$, the solution must satisfy the following constraints:

---

**Constraint 1 — Total Slot Coverage**

The total new slots added in zip $z$ (expansion across both age groups + new builds) must meet the total slot deficit:

$$\sum_{f \in \mathcal{F}_z} \!\left(x_f^{05} + x_f^{512}\right) + C_S\, y_z^S + C_M\, y_z^M + C_L\, y_z^L \;\geq\; \text{TotalDeficit}_z \qquad \forall\, z \in \mathcal{Z}$$

where $C_S = 100$, $C_M = 200$, $C_L = 400$.

---

**Constraint 2 — Under-5 Slot Coverage**

The total 0–5 slots added in zip $z$ (from facility expansions + 0–5 slots in new builds) must meet the under-5 deficit:

$$\sum_{f \in \mathcal{F}_z} x_f^{05} + n_z^{05} \;\geq\; \text{Under5Deficit}_z \qquad \forall\, z \in \mathcal{Z}$$

---

**Constraint 3 — New Build Slot Accounting**

The slots allocated to age groups cannot exceed the total capacity of newly built facilities:

$$n_z^{05} + n_z^{512} \;\leq\; C_S\, y_z^S + C_M\, y_z^M + C_L\, y_z^L \qquad \forall\, z \in \mathcal{Z}$$

---

**Constraint 4 — New Build Under-5 Cap**

The 0–5 slots from new builds cannot exceed the maximum under-5 capacity the chosen facilities can physically provide:

$$n_z^{05} \;\leq\; U_S\, y_z^S + U_M\, y_z^M + U_L\, y_z^L \qquad \forall\, z \in \mathcal{Z}$$

where $U_S = 50$, $U_M = 100$, $U_L = 200$ are the **maximum** under-5 slots a new Small / Medium / Large facility can hold.

---

**Constraint 5 — Expansion Upper Bound per Facility**

Each existing facility $f$ with current capacity $n_f$ may add at most 120% of its current size, capped at 500 additional slots regardless of starting size:

$$x_f^{05} + x_f^{512} \;\leq\; \min\!\left(1.2\, n_f,\; 500\right) \qquad \forall\, f \in \mathcal{F}_z,\; z \in \mathcal{Z}$$

> Note: the 500-slot cap applies to **slots added**, not to total resulting capacity. Facilities already above 500 slots are still eligible to expand by up to 120%.


In [ ]:
# ── Under-5 capacity caps for new facilities ─────────────────────────────────────────────
U_S = 50    # max 0-5 slots in a new small  facility (100 total)
U_M = 100   # max 0-5 slots in a new medium facility (200 total)
U_L = 200   # max 0-5 slots in a new large  facility (400 total)

# ── Pre-compute lookups ───────────────────────────────────────────────────────────────────
desert_df  = df[df['is_desert']].set_index('zipcode')
fac_by_zip = desert_facilities.groupby('zipcode')['facility_id'].apply(list).to_dict()

# ── Constraint 1: Total Slot Coverage ──────────────────────────────────────────────────
# Σ_f (x_05[f] + x_512[f]) + C_S·y_S[z] + C_M·y_M[z] + C_L·y_L[z] >= TotalDeficit_z
for z in desert_zips:
    facs = fac_by_zip.get(z, [])
    total_deficit = int(desert_df.loc[z, 'total_slot_deficit'])
    m.addConstr(
        gp.quicksum(x_05[f] + x_512[f] for f in facs)
        + CAP_S * y['S'][z] + CAP_M * y['M'][z] + CAP_L * y['L'][z]
        >= total_deficit,
        name=f"total_coverage_{z}"
    )

# ── Constraint 2: Under-5 Slot Coverage ───────────────────────────────────────────────
# Σ_f x_05[f] + n_05[z] >= Under5Deficit_z
for z in desert_zips:
    facs = fac_by_zip.get(z, [])
    under5_deficit = int(desert_df.loc[z, 'under5_slot_deficit'])
    m.addConstr(
        gp.quicksum(x_05[f] for f in facs) + n_05[z]
        >= under5_deficit,
        name=f"under5_coverage_{z}"
    )

# ── Constraint 3: New Build Slot Accounting ───────────────────────────────────────────
# n_05[z] + n_512[z] <= C_S·y_S[z] + C_M·y_M[z] + C_L·y_L[z]
for z in desert_zips:
    m.addConstr(
        n_05[z] + n_512[z]
        <= CAP_S * y['S'][z] + CAP_M * y['M'][z] + CAP_L * y['L'][z],
        name=f"new_build_cap_{z}"
    )

# ── Constraint 4: New Build Under-5 Cap ──────────────────────────────────────────────
# n_05[z] <= U_S·y_S[z] + U_M·y_M[z] + U_L·y_L[z]
for z in desert_zips:
    m.addConstr(
        n_05[z] <= U_S * y['S'][z] + U_M * y['M'][z] + U_L * y['L'][z],
        name=f"new_build_under5_cap_{z}"
    )

# ── Constraint 5: Expansion Upper Bound per Facility ───────────────────────────────
# x_05[f] + x_512[f] <= min(1.2 * n_f, 500)
# 1.2 * n_f : at most 120% of current capacity may be added
# 500        : absolute cap on slots added, regardless of starting size
# Facilities already above 500 slots are still eligible to expand by up to 120%.
for f, n_f in cap_lookup.items():
    ub = min(1.2 * n_f, 500)
    m.addConstr(x_05[f] + x_512[f] <= ub, name=f"expansion_cap_{f}")

m.update()
print("Constraints added:")
print(f"  Constraint 1 — Total slot coverage       : {len(desert_zips)} constraints")
print(f"  Constraint 2 — Under-5 coverage          : {len(desert_zips)} constraints")
print(f"  Constraint 3 — New build slot cap         : {len(desert_zips)} constraints")
print(f"  Constraint 4 — New build under-5 cap      : {len(desert_zips)} constraints")
print(f"  Constraint 5 — Expansion upper bound      : {len(cap_lookup)} constraints")
print(f"\nTotal constraints: {m.NumConstrs}")


Constraints added:
  Constraint 1 — Total slot coverage       : 179 constraints
  Constraint 2 — Under-5 coverage          : 179 constraints
  Constraint 3 — New build slot cap         : 179 constraints
  Constraint 4 — New build under-5 cap      : 179 constraints
  Constraint 5 — Expansion upper bound      : 7739 constraints

Total constraints: 39411


### Step 4: Solve the Model

In [ ]:
m.optimize()

if m.Status == GRB.OPTIMAL:
    total_cost  = m.ObjVal

    C_exp_val = sum(
        (10_000 + 200 * cap_lookup[f]) / cap_lookup[f] * d1[f].X
        + (20_000 + 200 * cap_lookup[f]) / cap_lookup[f] * d2[f].X
        for f in cap_lookup
    )
    C_build_val = sum(COST_S * y['S'][z].X + COST_M * y['M'][z].X + COST_L * y['L'][z].X for z in desert_zips)
    C_equip_val = EQUIP_COST * sum(
        sum(x_05[f].X for f in fac_by_zip.get(z, [])) + n_05[z].X
        for z in desert_zips
    )

    total_exp_05  = sum(x_05[f].X  for f in cap_lookup)
    total_exp_512 = sum(x_512[f].X for f in cap_lookup)
    n_cheap = sum(1 for f in cap_lookup if w[f].X < 0.5)   # facilities on cheap tier
    n_exp   = sum(1 for f in cap_lookup if w[f].X >= 0.5)  # facilities on expensive tier
    total_new_S   = sum(y['S'][z].X for z in desert_zips)
    total_new_M   = sum(y['M'][z].X for z in desert_zips)
    total_new_L   = sum(y['L'][z].X for z in desert_zips)
    total_new_facs = total_new_S + total_new_M + total_new_L
    total_n_05    = sum(n_05[z].X  for z in desert_zips)
    total_n_512   = sum(n_512[z].X for z in desert_zips)

    print("=" * 55)
    print("           OPTIMAL SOLUTION FOUND")
    print("=" * 55)
    print(f"\n  Total Minimum Cost                 : ${total_cost:>15,.2f}")
    print(f"  ├─ Expansion Cost                  : ${C_exp_val:>15,.2f}")
    print(f"  ├─ Construction Cost               : ${C_build_val:>15,.2f}")
    print(f"  └─ Equipment Surcharge             : ${C_equip_val:>15,.2f}")
    print(f"\n  Expansion slots added (0-5)        : {total_exp_05:>10,.0f}")
    print(f"  Expansion slots added (5-12)       : {total_exp_512:>10,.0f}")
    print(f"    ├─ facilities on cheap tier (<nf)  : {n_cheap:>10,}")
    print(f"    └─ facilities on exp   tier (≥nf) : {n_exp:>10,}")
    print(f"  New facilities built               : {total_new_facs:>10,.0f}")
    print(f"    ├─ Small  (100 slots)             : {total_new_S:>10,.0f}")
    print(f"    ├─ Medium (200 slots)             : {total_new_M:>10,.0f}")
    print(f"    └─ Large  (400 slots)             : {total_new_L:>10,.0f}")
    print(f"  New build slots (0-5)              : {total_n_05:>10,.0f}")
    print(f"  New build slots (5-12)             : {total_n_512:>10,.0f}")

elif m.Status == GRB.INFEASIBLE:
    print("Model is INFEASIBLE — no solution satisfies all constraints.")
    m.computeIIS()
    m.write("infeasible.ilp")
    print("IIS written to infeasible.ilp for diagnosis.")
else:
    print(f"Solver status: {m.Status}")

           OPTIMAL SOLUTION FOUND

  Total Minimum Cost                 : $ 166,924,983.32
  ├─ Expansion Cost                  : $  62,116,283.32
  ├─ Construction Cost               : $  70,175,000.00
  └─ Equipment Surcharge             : $  34,633,700.00

  Expansion slots added (0-5)        :    224,675
  Expansion slots added (5-12)       :          3
    ├─ facilities on cheap tier (<nf)  :      7,433
    └─ facilities on exp   tier (≥nf) :        306
  New facilities built               :        611
    ├─ Small  (100 slots)             :          1
    ├─ Medium (200 slots)             :          2
    └─ Large  (400 slots)             :        608
  New build slots (0-5)              :    121,662
  New build slots (5-12)             :          0


## Save Results


In [ ]:
import os, json


RESULTS_DIR = '../data/results/task_1'
os.makedirs(RESULTS_DIR, exist_ok=True)

if m.Status == GRB.OPTIMAL:

    # ── 1. Summary (scalars) ──────────────────────────────────────────────────
    summary = {
        'total_cost':     total_cost,
        'C_exp':          C_exp_val,
        'C_build':        C_build_val,
        'C_equip':        C_equip_val,
        'expansion_slots_05':  total_exp_05,
        'expansion_slots_512': total_exp_512,
        'new_facs_S':     total_new_S,
        'new_facs_M':     total_new_M,
        'new_facs_L':     total_new_L,
        'new_build_slots_05':  total_n_05,
        'new_build_slots_512': total_n_512,
        'facilities_cheap_tier': n_cheap,
        'facilities_exp_tier':   n_exp,
    }
    with open(os.path.join(RESULTS_DIR, 'summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)

    # ── 2. Per-facility expansion results ─────────────────────────────────────
    zip_of = desert_facilities.set_index('facility_id')['zipcode'].to_dict()
    rows = []
    for f, n_f in cap_lookup.items():
        x05  = x_05[f].X
        x512 = x_512[f].X
        xtot = x05 + x512
        tier = 'expensive' if w[f].X >= 0.5 else 'cheap'
        cheap_rate = (10_000 + 200 * n_f) / n_f
        exp_rate   = (20_000 + 200 * n_f) / n_f
        cost = cheap_rate * d1[f].X + exp_rate * d2[f].X
        rows.append({
            'facility_id':    f,
            'zipcode':        zip_of.get(f),
            'current_slots':  n_f,
            'x_05':           round(x05),
            'x_512':          round(x512),
            'x_total':        round(xtot),
            'tier':           tier,
            'expansion_cost': round(cost, 2),
        })
    expansions_df = pd.DataFrame(rows)
    expansions_df.to_csv(os.path.join(RESULTS_DIR, 'expansions.csv'), index=False)

    # ── 3. Per-zip new build results ──────────────────────────────────────────
    desert_df_idx = df[df['is_desert']].set_index('zipcode')
    build_rows = []
    for z in desert_zips:
        yS = round(y['S'][z].X)
        yM = round(y['M'][z].X)
        yL = round(y['L'][z].X)
        build_rows.append({
            'zipcode':           z,
            'demand':            desert_df_idx.loc[z, 'demand'],
            'total_deficit':     int(desert_df_idx.loc[z, 'total_slot_deficit']),
            'under5_deficit':    int(desert_df_idx.loc[z, 'under5_slot_deficit']),
            'new_facs_S':        yS,
            'new_facs_M':        yM,
            'new_facs_L':        yL,
            'total_new_facs':    yS + yM + yL,
            'new_slots_total':   yS*CAP_S + yM*CAP_M + yL*CAP_L,
            'n_05':              round(n_05[z].X),
            'n_512':             round(n_512[z].X),
            'exp_slots_05':      round(sum(x_05[f].X  for f in fac_by_zip.get(z, []))),
            'exp_slots_512':     round(sum(x_512[f].X for f in fac_by_zip.get(z, []))),
        })
    builds_df = pd.DataFrame(build_rows)
    builds_df.to_csv(os.path.join(RESULTS_DIR, 'zip_results.csv'), index=False)

    print(f"Results saved to {RESULTS_DIR}/")
    print(f"  summary.json      — scalar cost breakdown")
    print(f"  expansions.csv    — {len(expansions_df[expansions_df['x_total']>0])} facilities expanded")
    print(f"  zip_results.csv   — {len(builds_df)} desert zip codes")
else:
    print("No optimal solution to save.")

Results saved to ../data/results/task_1/
  summary.json      — scalar cost breakdown
  expansions.csv    — 1362 facilities expanded
  zip_results.csv   — 179 desert zip codes
